# CredResolve Data Analyst Assignment

## Analytical Investigation

This notebook presents the consolidated analytical investigation performed on the validated Golden layer.

The analysis covers portfolio recovery, operational performance, data quality, forensic findings, statistical limitations, the reported 11% improvement, counterfactual comparison, and the ₹10 Cr investment decision.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/golden")
REPORTS = Path("../reports")

accounts = pd.read_csv(BASE / "accounts.csv")
payments = pd.read_csv(BASE / "payments.csv")
calls = pd.read_csv(BASE / "calls.csv")

successful_payments = payments[
    payments["payment_status"].astype(str).str.upper().eq("SUCCESS")
].copy()

print("Accounts:", len(accounts))
print("Calls:", len(calls))
print("Payments:", len(payments))
print("Successful payments:", len(successful_payments))


Accounts: 30000
Calls: 90079
Payments: 25000
Successful payments: 17534


## 1. Portfolio Recovery

The account is the canonical analytical entity. Recovery is based on successful payments attributed through `account_id`.


In [2]:
recovered_amount = successful_payments["amount"].sum()
outstanding_amount = accounts["outstanding_amount"].sum()

recovery_rate = recovered_amount / outstanding_amount * 100

portfolio_summary = pd.DataFrame({
    "metric": [
        "Accounts",
        "Outstanding amount",
        "Successful payment amount",
        "Successful payments",
        "Observed recovery rate"
    ],
    "value": [
        len(accounts),
        outstanding_amount,
        recovered_amount,
        len(successful_payments),
        recovery_rate
    ]
})

portfolio_summary


,metric,value
0,Accounts,3.000000e+04
1,Outstanding amount,1.048904e+10
2,Successful payment amount,1.315584e+09
3,Successful payments,1.753400e+04
4,Observed recovery rate,1.254247e+01


## 2. Data Quality and Validation

The cleaned analytical layer passed 49 of 49 validation checks.

Cleaning decisions were designed to remove unambiguous duplication while retaining ambiguous records with explicit quality flags.


In [3]:
validation = pd.read_csv(
    REPORTS / "final_cleaned_layer_validation.csv"
)

validation.head()


,check,actual,expected,status
0,accounts_row_count,30000,30000,PASS
1,accounts_exact_duplicates,0,0,PASS
2,borrowers_row_count,30000,30000,PASS
3,borrowers_exact_duplicates,0,0,PASS
4,agents_row_count,30000,30000,PASS


## 3. Monthly Performance

Monthly activity is descriptive because historical eligible outstanding-balance denominators are unavailable.

Therefore recovered amount is not presented as a historical recovery-rate measure.


In [4]:
monthly = pd.read_csv(
    REPORTS / "monthly_performance.csv"
)

monthly[[
    "month",
    "recovered_amount",
    "successful_payments",
    "calls",
    "answered_calls",
    "answer_rate_pct",
    "recovered_per_call"
]]


,month,recovered_amount,successful_payments,calls,answered_calls,answer_rate_pct,recovered_per_call
0,2025-12,0.000000e+00,0.0,1,0,0.0000,0.0000
1,2026-01,1.872291e+08,2464.0,12698,2545,20.0425,14744.7730
2,2026-02,1.701425e+08,2268.0,11580,2282,19.7064,14692.7853
3,2026-03,1.889124e+08,2524.0,12857,2570,19.9891,14693.3479
4,2026-04,1.751380e+08,2406.0,12258,2366,19.3017,14287.6524
5,2026-05,1.842503e+08,2449.0,12754,2594,20.3387,14446.4700
6,2026-06,1.755597e+08,2366.0,12137,2476,20.4004,14464.8370
7,2026-07,1.872423e+08,2441.0,12583,2433,19.3356,14880.5742
8,2026-08,4.710970e+07,616.0,3211,630,19.6201,14671.3470


## 4. Portfolio Drivers

Recovery varies across DPD, risk, account status, and loan type.

These differences are descriptive and do not establish causality.


In [5]:
driver_summary = pd.read_csv(
    REPORTS / "driver_summary.csv"
)

driver_summary


,driver,best_segment,best_recovery_rate_pct,worst_segment,worst_recovery_rate_pct
0,risk_segment,MEDIUM,12.575628,NPA,12.497968
1,dpd_band,31-60,13.203362,1-30,12.298577
2,status,WRITEOFF,12.634267,CLOSED,12.354541
3,loan_type,CONSUMER,12.878058,PERSONAL,12.247599
4,borrower_quality,resolved,12.566867,unresolved,12.285955


In [6]:
for name in [
    "recovery_by_dpd.csv",
    "recovery_by_risk.csv",
    "recovery_by_account_status.csv",
    "recovery_by_loan_type.csv"
]:
    print("\n", name)
    display(pd.read_csv(REPORTS / name))



 recovery_by_dpd.csv


,dpd_band,accounts,outstanding_amount,recovered_amount,accounts_with_payment,successful_payment_count,recovery_rate_pct,payment_account_rate_pct
0,31-60,5514,1.914490e+09,2.527771e+08,2524,3352.0,13.203362,45.774392
1,61-90,5468,1.895868e+09,2.387357e+08,2436,3164.0,12.592417,44.550110
2,91-180,5453,1.896357e+09,2.350695e+08,2385,3124.0,12.395843,43.737392
3,0,2685,9.482186e+08,1.174619e+08,1220,1587.0,12.387642,45.437616
4,1-30,10880,3.834101e+09,4.715398e+08,4719,6307.0,12.298577,43.373162



 recovery_by_risk.csv


,risk_segment,accounts,outstanding_amount,recovered_amount,accounts_with_payment,successful_payment_count,recovery_rate_pct,payment_account_rate_pct
0,MEDIUM,7533,2.628179e+09,3.305101e+08,3335,4422.0,12.575628,44.271870
1,LOW,7513,2.633311e+09,3.308578e+08,3381,4467.0,12.564328,45.001997
2,HIGH,7552,2.646183e+09,3.315982e+08,3300,4364.0,12.531192,43.697034
3,NPA,7402,2.581362e+09,3.226178e+08,3268,4281.0,12.497968,44.150230



 recovery_by_account_status.csv


,status,accounts,outstanding_amount,recovered_amount,accounts_with_payment,successful_payment_count,recovery_rate_pct,payment_account_rate_pct
0,WRITEOFF,7479,2.597956e+09,3.282327e+08,3347,4370.0,12.634267,44.751972
1,ACTIVE,7539,2.637397e+09,3.329433e+08,3315,4404.0,12.623935,43.971349
2,PAID,7486,2.617540e+09,3.287247e+08,3297,4379.0,12.558537,44.042212
3,CLOSED,7496,2.636143e+09,3.256834e+08,3325,4381.0,12.354541,44.356990



 recovery_by_loan_type.csv


,loan_type,accounts,outstanding_amount,recovered_amount,accounts_with_payment,successful_payment_count,recovery_rate_pct,payment_account_rate_pct
0,CONSUMER,5930,2.073113e+09,2.669767e+08,2685,3566.0,12.878058,45.278246
1,CREDIT_CARD,6080,2.126677e+09,2.686432e+08,2690,3543.0,12.632065,44.243421
2,AUTO,6079,2.135301e+09,2.679332e+08,2700,3559.0,12.547796,44.415200
3,BNPL,5928,2.058915e+09,2.554400e+08,2580,3415.0,12.406537,43.522267
4,PERSONAL,5983,2.095030e+09,2.565909e+08,2629,3451.0,12.247599,43.941167


## 5. Channel and Campaign Performance

Observed channel and campaign differences are used as operational hypotheses.

They are not interpreted as causal effects because portfolio mix, contact selection, agent allocation, and other confounders remain relevant.


In [7]:
channel = pd.read_csv(
    REPORTS / "channel_performance.csv"
)

campaign = pd.read_csv(
    REPORTS / "campaign_recovery_performance.csv"
)

display(channel)
display(campaign.head(20))


,channel,activity,successful_outcomes,success_rate_pct
0,WHATSAPP,60000,29919,49.865000
1,SMS,45000,11219,24.931111
2,VOICE,90079,17896,19.867006
3,FIELD,25000,4205,16.820000


,campaign_id,campaign_name,channel,strategy_version,call_volume,answered_calls,unique_accounts,accounts_with_successful_payment,successful_payment_amount,answer_rate_pct,payment_account_rate_pct,payment_per_call
0,CMP0000006,BOUNCE,MIXED,legacy,750,149,740,354,37686789.59,19.866667,47.837838,50249.052787
1,CMP0000036,30DPD_W1,SMS,v1,792,172,781,366,37448151.55,21.717172,46.862996,47283.019634
2,CMP0000116,DIGITAL_FOLLOWUP,VOICE,v1,816,162,804,386,36874603.90,19.852941,48.009950,45189.465564
3,CMP0000055,BOUNCE,MIXED,v3,756,148,750,340,36718583.36,19.576720,45.333333,48569.554709
4,CMP0000100,BOUNCE,WHATSAPP,v3,792,148,785,359,36547359.96,18.686869,45.732484,46145.656515
5,CMP0000112,60DPD_INTENT,MIXED,v3,784,171,776,339,36134241.83,21.811224,43.685567,46089.594171
6,CMP0000072,30DPD_W1,WHATSAPP,v2,790,154,777,359,36093921.40,19.493671,46.203346,45688.508101
7,CMP0000098,DIGITAL_FOLLOWUP,WHATSAPP,legacy,773,156,768,369,35952701.90,20.181113,48.046875,46510.610479
8,CMP0000114,60DPD_INTENT,SMS,legacy,822,163,805,379,35890700.27,19.829684,47.080745,43662.652397
9,CMP0000019,NPA_RECOVERY,WHATSAPP,v1,763,149,748,352,35635736.99,19.528178,47.058824,46704.766697


## 6. Forensic Analysis

The forensic investigation covered payment attribution, vendor performance, calling time, and attempt frequency.

Payment attribution is sufficiently defined at account level through:

`payment_id -> account_id`

Successful payment attribution coverage is 100% for the cleaned successful-payment population.


In [8]:
forensic = pd.read_csv(
    REPORTS / "forensic_attempt_frequency.csv"
)

vendor = pd.read_csv(
    REPORTS / "forensic_vendor_performance.csv"
)

calling_time = pd.read_csv(
    REPORTS / "forensic_calling_time.csv"
)

display(forensic)
display(vendor)
display(calling_time)


,attempt_band,accounts,paying_accounts,payment_account_rate_pct
0,1,2124,902,42.467043
1,2,4426,1975,44.622684
2,3,5951,2640,44.362292
3,4-5,10553,4677,44.319151
4,6-10,6310,2818,44.659271
5,11+,87,32,36.781609


,vendor_id,calls,answered_calls
0,VND0000001,6025,6031
1,VND0000002,5934,5941
2,VND0000003,5964,5966
3,VND0000004,5925,5930
4,VND0000005,6048,6055
5,VND0000006,5896,5903
6,VND0000007,6069,6074
7,VND0000008,5911,5917
8,VND0000009,5972,5976
9,VND0000010,6163,6167


,hour,calls,answered_calls
0,0,3629,3633
1,1,3745,3747
2,2,3878,3884
3,3,3775,3777
4,4,3697,3700
5,5,3797,3800
6,6,3815,3819
7,7,3793,3797
8,8,3700,3704
9,9,3699,3702


## 7. Statistical Investigation

The analysis identified portfolio-mix effects, selection bias, survivorship concerns, attribution-window limitations, Simpson's-paradox risk, and boundary-period effects.

These limitations prevent a causal interpretation of aggregate recovery movement.


In [9]:
print(
    (REPORTS / "statistical_investigation_findings.md").read_text(
        encoding="utf-8"
    )
)


# Statistical Investigation Findings

## Portfolio Mix

Recovery performance differs across risk, DPD, account-status, and loan-type segments.
Therefore aggregate recovery can change when the composition of the handled portfolio changes.

## Cohort Effects

The available data does not provide historical monthly eligible balances for comparable cohorts.
A controlled before-and-after cohort recovery rate therefore cannot be reconstructed.

## Selection Bias

Accounts receiving calls, PTPs, or other interventions are not necessarily a random sample of the portfolio.
Higher activity may reflect accounts that were easier or harder to recover.

## Survivorship Bias

Accounts remaining active in later periods may differ systematically from accounts resolved earlier.
The current data does not provide sufficient historical snapshots to fully reconstruct this effect.

## Simpson's Paradox

Aggregate recovery can differ from segment-level recovery because the portfolio mix changes.
Segment-level 

## 8. Reported 11% Improvement

The reported 11% improvement is classified as **UNVERIFIED**.

The principal limitation is the absence of historical eligible outstanding-balance denominators.

The available data supports descriptive recovery comparisons but does not establish a causal or mix-adjusted 11% improvement.


In [10]:
verification = pd.read_csv(
    REPORTS / "improvement_verification.csv"
)

verification


,risk_segment,accounts,outstanding_amount,portfolio_share_pct,recovered_amount,successful_payments,paying_accounts,observed_recovery_rate_pct,dimension,dpd_band,status,loan_type
0,HIGH,7552,2.646183e+09,25.228083,3.315982e+08,4364,3300,12.531192,risk_segment,NaN,NaN,NaN
1,LOW,7513,2.633311e+09,25.105370,3.308578e+08,4467,3381,12.564328,risk_segment,NaN,NaN,NaN
2,MEDIUM,7533,2.628179e+09,25.056446,3.305101e+08,4422,3335,12.575628,risk_segment,NaN,NaN,NaN
3,NPA,7402,2.581362e+09,24.610101,3.226178e+08,4281,3268,12.497968,risk_segment,NaN,NaN,NaN
4,NaN,2685,9.482186e+08,9.040094,1.174619e+08,1587,1220,12.387642,dpd_band,0,NaN,NaN
5,NaN,10880,3.834101e+09,36.553414,4.715398e+08,6307,4719,12.298577,dpd_band,1-30,NaN,NaN
6,NaN,5514,1.914490e+09,18.252301,2.527771e+08,3352,2524,13.203362,dpd_band,31-60,NaN,NaN
7,NaN,5468,1.895868e+09,18.074764,2.387357e+08,3164,2436,12.592417,dpd_band,61-90,NaN,NaN
8,NaN,5453,1.896357e+09,18.079427,2.350695e+08,3124,2385,12.395843,dpd_band,91-180,NaN,NaN
9,NaN,7539,2.637397e+09,25.144322,3.329433e+08,4404,3315,12.623935,status,NaN,ACTIVE,NaN


## 9. Counterfactual Benchmark

Answered-call accounts are treated as the observational treatment group and accounts without answered calls as the comparison group.

The observed difference is a benchmark, not a causal treatment effect.


In [11]:
counterfactual = pd.read_csv(
    REPORTS / "counterfactual_analysis.csv"
) if (REPORTS / "counterfactual_analysis.csv").exists() else None

if counterfactual is not None:
    display(counterfactual)
else:
    print(
        (REPORTS / "counterfactual_analysis_findings.md").read_text(
            encoding="utf-8"
        )
    )


# Counterfactual Analysis

## Design

The analysis uses an observational treatment and comparison framework.

Treatment accounts are accounts with at least one answered call.

Comparison accounts are accounts without an answered call.

Accounts are compared within strata defined by DPD band, risk segment,
account status, and loan type.

## Observed Comparison

Treatment accounts: 13,535

Comparison accounts: 16,465

Treatment payment-account rate: 44.70%

Comparison payment-account rate: 43.94%

Observed difference: +0.76 percentage points.

Matched strata: 400

## Interpretation

The treatment group has a higher or lower observed payment rate than the
comparison group after restricting the comparison to common portfolio
strata.

This difference is an observational association and must not be interpreted
as a causal treatment effect.

Accounts receiving answered calls were selected operationally and may differ
from accounts without answered calls in unobserved characteristics,
collecti

## 10. ₹10 Cr Investment Decision

The recommended approach is a controlled collections and contactability optimization pilot rather than an unconditional ₹10 Cr rollout.

The observed evidence is insufficient to justify assuming the reported 11% improvement.


In [12]:
investment = pd.read_csv(
    REPORTS / "investment_scenarios.csv"
)

recommendation = pd.read_csv(
    REPORTS / "investment_recommendation.csv"
)

display(investment)
display(recommendation)


,scenario,assumed_incremental_payment_rate,eligible_accounts,incremental_accounts_recovered,average_recovery_per_incremental_account,incremental_recovery,investment,net_value,roi_pct,benefit_cost_ratio
0,Downside,0.0038,16465,62.567,99035.22769,6.196337e+06,100000000,-9.380366e+07,-93.803663,0.061963
1,Base,0.0076,16465,125.134,99035.22769,1.239267e+07,100000000,-8.760733e+07,-87.607326,0.123927
2,Upside,0.0152,16465,250.268,99035.22769,2.478535e+07,100000000,-7.521465e+07,-75.214652,0.247853


,recommended_investment,investment_amount,priority_channel,observed_best_channel_success_rate_pct,answered_accounts,unanswered_accounts,observed_answered_vs_unanswered_payment_difference_pct_points,base_assumed_incremental_payment_rate,break_even_incremental_accounts,break_even_incremental_rate_pct,decision,confidence
0,Targeted contactability and collections optimi...,100000000,WHATSAPP,49.865,13535,16465,0.763308,0.0076,1009.741708,6.132655,Proceed only as a controlled pilot with measur...,Moderate-to-low


## Final Conclusion

The portfolio shows measurable recovery activity and meaningful variation across operational and portfolio dimensions.

However, the reported 11% improvement cannot be independently validated from the supplied data because historical eligible balances and sufficiently controlled cohorts are unavailable.

The recommended business decision is therefore a controlled pilot with a predefined holdout, explicit attribution window, eligible-balance measurement, and a break-even threshold before scaling.

The analysis distinguishes observed relationships from causal claims and avoids presenting unverified improvement as fact.
